In [12]:
# -------------------------------------------------------------------------
# Step 1: Environment Setup & Gemini Client Initialization
# -------------------------------------------------------------------------
import os
import requests
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Load environment variables from .env file
load_dotenv()

# Initialize the Google GenAI client (reads GEMINI_API_KEY from environment)
client = genai.Client()

print("✅ Gemini Client successfully initialized!")
print("OpenWeather API Key configured:", bool(os.getenv("OPENWEATHERMAP_API_KEY") or os.getenv("OPENWEATHER_API_KEY")))

✅ Gemini Client successfully initialized!
OpenWeather API Key configured: True


In [13]:
# -------------------------------------------------------------------------
# Step 2: Define the OpenWeather Tool Function
# -------------------------------------------------------------------------
def get_weather(city: str) -> dict:
    """Fetches the current weather details and temperature for a given city using OpenWeatherMap.
    
    Args:
        city: The name of the city (e.g., 'Kathmandu', 'London', 'New York').
    """
    api_key = os.getenv("OPENWEATHERMAP_API_KEY") or os.getenv("OPENWEATHER_API_KEY")
    if not api_key:
        return {"error": "OPENWEATHERMAP_API_KEY is not configured in .env."}
        
    print(f"\n[TOOL CALL] Querying OpenWeather for: '{city}'...")
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city,
        "appid": api_key,
        "units": "metric"  # metric returns temperature in Celsius
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        
        if response.status_code == 200:
            return {
                "city": data.get("name", city),
                "temperature": data["main"]["temp"],
                "unit": "Celsius",
                "condition": data["weather"][0]["description"],
                "humidity": data["main"]["humidity"]
            }
        else:
            error_msg = data.get("message", "Unknown error")
            print(f"[TOOL ERROR] OpenWeather error for '{city}': {error_msg}")
            return {"error": error_msg, "city": city}
    except Exception as e:
        print(f"[TOOL EXCEPTION] Network/request error for '{city}': {e}")
        return {"error": str(e), "city": city}

In [14]:
# -------------------------------------------------------------------------
# Step 3: Configure the Weather Agent with Sequential Reasoning Instructions
# -------------------------------------------------------------------------
weather_system_instruction = (
    "You are an analytical weather agent. When given multiple locations:\n"
    "1. You must call `get_weather` for each location sequentially (one by one).\n"
    "2. Extract and record each location's temperature in Celsius.\n"
    "3. Calculate the exact mathematical average of all temperatures: (T1 + T2 + T3) / 3.\n"
    "4. Present a final clean summary containing:\n"
    "   - A bulleted list or table of each city, its current temperature, and conditions\n"
    "   - The step-by-step mathematical average calculation\n"
    "   - The final average temperature clearly highlighted in Celsius."
)

# Initialize agent chat with Automatic Function Calling (AFC)
weather_agent = client.chats.create(
    model="gemini-3.5-flash-lite",
    config=types.GenerateContentConfig(
        system_instruction=weather_system_instruction,
        tools=[get_weather],
        temperature=0.0
    )
)

In [15]:
# -------------------------------------------------------------------------
# Step 4: Run the Sequential Weather Agent across 3 Locations
# -------------------------------------------------------------------------
user_prompt = (
    "Retrieve the current weather for Kathamndu, London, and New York sequentially. "
    "Collect their temperatures and calculate the average temperature across all three."
)

print(f"User Query: {user_prompt}\n")
print("--- Weather Agent Execution ---")

# Gemini sequentially calls get_weather for Tokyo -> London -> New York and synthesizes the average
response = weather_agent.send_message(user_prompt)

print("\n--- Final Agent Report ---")
print(response.text)

User Query: Retrieve the current weather for Kathamndu, London, and New York sequentially. Collect their temperatures and calculate the average temperature across all three.

--- Weather Agent Execution ---

[TOOL CALL] Querying OpenWeather for: 'Kathamndu'...
[TOOL ERROR] OpenWeather error for 'Kathamndu': city not found

[TOOL CALL] Querying OpenWeather for: 'Kathmandu'...

[TOOL CALL] Querying OpenWeather for: 'London'...

[TOOL CALL] Querying OpenWeather for: 'New York'...

--- Final Agent Report ---
Here is the weather data and temperature analysis for the requested cities:

### **City Weather Summary**
* **Kathmandu**: 18.97 °C (Moderate rain)
* **London**: 17.44 °C (Scattered clouds)
* **New York**: 19.47 °C (Overcast clouds)

---

### **Average Temperature Calculation**
1. **Sum of temperatures:** $18.97 + 17.44 + 19.47 = 55.88$
2. **Number of locations:** $3$
3. **Division:** $55.88 \div 3 = 18.6267$

---

### **Final Result**
* **Average Temperature:** **18.63 °C**
